In [ ]:
import os
import time
import json
import warnings
from typing import Tuple, Dict, Any, List

import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Tắt cảnh báo và cấu hình môi trường
warnings.filterwarnings('ignore')
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
torch.set_num_threads(8)

# ==========================================
# 1. DATA PREPROCESSING & UTILS
# ==========================================
def load_and_preprocess_data(dataset_name: str) -> Tuple[np.ndarray, np.ndarray]:
    """Tải và tiền xử lý dữ liệu tabular (chuẩn hóa MinMaxScaler)."""
    if dataset_name == 'breast_cancer':
        data = load_breast_cancer()
        X, y = data.data[:, :8], data.target
    elif dataset_name == 'banknote':
        data = fetch_openml(name='banknote-authentication', version=1, as_frame=False, parser='auto')
        X, y = data.data, data.target
        y = LabelEncoder().fit_transform(y)
    else:
        raise ValueError(f"Dataset {dataset_name} chưa được hỗ trợ.")
        
    scaler = MinMaxScaler(feature_range=(0, np.pi))
    X = scaler.fit_transform(X)
    return X, y

def apply_pca_if_needed(X_train: np.ndarray, X_test: np.ndarray, emb_type: str, seed: int) -> Tuple[np.ndarray, np.ndarray, bool]:
    """Tự động nén PCA xuống 8 chiều nếu dữ liệu quá lớn cho Angle/IQP Embedding."""
    if emb_type in ['angle', 'iqp'] and X_train.shape[1] > 8:
        pca = PCA(n_components=8, random_state=seed)
        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)
        
        # Chuẩn hóa lại sau khi PCA
        scaler = MinMaxScaler(feature_range=(0, np.pi))
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        return X_train, X_test, True
    return X_train, X_test, False

def calculate_required_qubits(n_features: int, embedding_type: str) -> int:
    """Tính số lượng Qubits cần thiết dựa trên phương pháp Embedding."""
    if embedding_type == 'amplitude':
        return int(np.ceil(np.log2(n_features)))
    return n_features

# ==========================================
# 2. QUANTUM MODEL ARCHITECTURE
# ==========================================
def create_quantum_layer(n_qubits: int, embedding_type: str, n_layers: int):
    """Tạo Quantum Circuit (QNode) bằng PennyLane."""
    dev = qml.device("default.qubit", wires=n_qubits) 
    
    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def quantum_circuit(inputs, weights):
        if embedding_type == 'angle':
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation='Y')
        elif embedding_type == 'amplitude':
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True, pad_with=0.)
        elif embedding_type == 'iqp':
            qml.IQPEmbedding(inputs, wires=range(n_qubits))
            
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return qml.expval(qml.PauliZ(0))
        
    return quantum_circuit

class HybridQMLModel(nn.Module):
    """Mô hình Lai (Hybrid) kết hợp PyTorch và PennyLane."""
    def __init__(self, n_qubits: int, embedding_type: str, n_layers: int):
        super().__init__()
        weight_shape = {"weights": (n_layers, n_qubits, 3)} 
        self.qlayer = qml.qnn.TorchLayer(
            create_quantum_layer(n_qubits, embedding_type, n_layers), weight_shape
        )
        self.fc = nn.Linear(1, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.sigmoid(self.fc(self.qlayer(x).unsqueeze(1)))

# ==========================================
# 3. TRAINING & EVALUATION
# ==========================================
def train_single_run(
    X_train: np.ndarray, y_train: np.ndarray, 
    X_test: np.ndarray, y_test: np.ndarray, 
    emb_type: str, n_layers: int, epochs: int, lr: float, device: torch.device
) -> Dict[str, Any]:
    """Thực thi toàn bộ quá trình huấn luyện và đánh giá cho 1 Run cụ thể."""
    # 1. Chuyển đổi dữ liệu sang Tensor
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)
    
    n_q = calculate_required_qubits(X_tr.shape[1], emb_type)
    
    # 2. Khởi tạo Model, Optimizer, Loss
    model = HybridQMLModel(n_q, emb_type, n_layers).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    
    run_te_acc = []
    start_time = time.time() 
    
    # 3. Vòng lặp Epochs
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_tr), y_tr)
        loss.backward()
        optimizer.step()

        # Đánh giá sau mỗi epoch để vẽ biểu đồ Convergence
        with torch.no_grad():
            model.eval()
            preds_te = (model(X_te).cpu() > 0.5).float().numpy()
            run_te_acc.append(accuracy_score(y_test, preds_te))
            
    run_time = time.time() - start_time
    
    # 4. Tính toán Metrics cuối cùng (F1, AUC)
    with torch.no_grad():
        model.eval()
        final_probs = model(X_te).cpu().numpy().flatten()
        final_preds = (final_probs > 0.5).astype(float)
    
    final_f1 = f1_score(y_test, final_preds, average='weighted')
    try:
        final_auc = roc_auc_score(y_test, final_probs)
    except ValueError:
        final_auc = np.nan
        
    return {
        'qubits': n_q,
        'time': run_time,
        'accuracy': run_te_acc[-1],
        'f1': final_f1,
        'auc': final_auc,
        'convergence_history': run_te_acc
    }

# ==========================================
# 4. MAIN
# ==========================================
# --- Cấu hình ---
DATASETS = ['banknote', 'breast_cancer']
EMBEDDING_TYPES = ['angle', 'amplitude', 'iqp'] 
NUM_RUNS = 50
EPOCHS = 100 
INITIAL_LR = 0.1 
N_LAYERS = 5 
CSV_FILENAME = 'angle_benchmark_results.csv'
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Bắt đầu Training trên thiết bị: {device.type.upper()}")

# Cố định random seeds cho tất cả các runs để tái lập kết quả
np.random.seed(42)
seeds = np.random.randint(0, 10000, size=NUM_RUNS)
all_results_data = [] 

for dataset_name in DATASETS:
    X_raw, y_raw = load_and_preprocess_data(dataset_name)
    
    for emb_type in EMBEDDING_TYPES:
        print(f"\n Đang chạy: {dataset_name.upper()} | Embedding: {emb_type.upper()}")
        
        for i in range(NUM_RUNS):
            # 1. Split Data
            X_train, X_test, y_train, y_test = train_test_split(
                X_raw, y_raw, test_size=0.3, random_state=seeds[i]
            )
            
            # 2. PCA Compression 
            X_train, X_test, is_compressed = apply_pca_if_needed(X_train, X_test, emb_type, seeds[i])
            if i == 0 and is_compressed:
                print(f"   Dữ liệu lớn. Đã tự động nén PCA xuống 8 chiều cho toàn bộ {NUM_RUNS} runs!")

            # 3. Train & Trích xuất kết quả
            metrics = train_single_run(
                X_train, y_train, X_test, y_test, 
                emb_type, N_LAYERS, EPOCHS, INITIAL_LR, device
            )
            
            print(f"   -> Run {i+1:02d}/{NUM_RUNS} | Qubits: {metrics['qubits']} | "
                    f"Time: {metrics['time']:.2f}s | Acc: {metrics['accuracy']:.4f} | "
                    f"F1: {metrics['f1']:.4f} | AUC: {metrics['auc']:.4f}")

            # 4. Lưu log
            all_results_data.append({
                'Dataset': dataset_name.upper(),
                'Embedding': emb_type.upper(),
                'Seed': seeds[i],                              
                'Accuracy': metrics['accuracy'],                    
                'F1_Score': metrics['f1'],                        
                'AUC_Score': metrics['auc'],                      
                'Time_Seconds': metrics['time'],                      
                'Convergence_History': json.dumps(metrics['convergence_history'])  
            })

# --- Lưu ra file CSV ---
df_export = pd.DataFrame(all_results_data)
df_export.to_csv(CSV_FILENAME, index=False)
print(f"\nĐÃ XUẤT THÀNH CÔNG DỮ LIỆU RA FILE: {CSV_FILENAME}")

In [ ]:
import json
import ast
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cài đặt theme chung cho các biểu đồ đẹp hơn
sns.set_theme(style="whitegrid")

def parse_history_string(history_str: str) -> list:
    """Hàm helper để parse chuỗi string thành list (xử lý json hoặc ast)."""
    try:
        return json.loads(history_str)
    except (json.JSONDecodeError, TypeError):
        return ast.literal_eval(history_str)

def plot_tradeoff_detailed(df_ds: pd.DataFrame, dataset_name: str, save_dir: Path) -> None:
    """Vẽ biểu đồ ma trận đánh đổi giữa Thời gian và Độ chính xác."""
    plt.figure(figsize=(11, 6))
    
    # 1. Vẽ các điểm phân tán
    sns.scatterplot(
        x='Time_Seconds', y='Accuracy', hue='Embedding', style='Embedding',
        data=df_ds, palette='tab10', s=100, alpha=0.8, zorder=3
    )
    
    # 2. Đưa legend ra ngoài
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    
    # 3. Tính trục trung tâm & vẽ ranh giới (4 góc)
    mean_time = df_ds['Time_Seconds'].mean()
    mean_acc = df_ds['Accuracy'].mean()
    
    plt.axvline(x=mean_time, color='black', linestyle='--', linewidth=1.2, alpha=0.4, zorder=1)
    plt.axhline(y=mean_acc, color='black', linestyle='--', linewidth=1.2, alpha=0.4, zorder=1)
    
    plt.xlabel('Thời gian huấn luyện (Giây)', fontsize=12)
    plt.ylabel('Độ chính xác (Accuracy)', fontsize=12)
    
    output_path = save_dir / f'fig_tradeoff_matrix_{dataset_name}.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_convergence_detailed(df_ds: pd.DataFrame, dataset_name: str, save_dir: Path) -> None:
    """Vẽ biểu đồ hội tụ (Convergence) với dải băng độ lệch chuẩn (Shadow)."""
    plt.figure(figsize=(12, 7))
    
    embeddings = df_ds['Embedding'].unique()
    colors = sns.color_palette('tab10', n_colors=len(embeddings))
    
    for idx, emb in enumerate(embeddings):
        df_emb = df_ds[df_ds['Embedding'] == emb]
        
        # Sử dụng hàm helper để parse dữ liệu nhanh và an toàn gọn gàng
        histories = np.array([parse_history_string(h) for h in df_emb['Convergence_History']])
            
        epochs = range(1, histories.shape[1] + 1)
        mean_h = np.mean(histories, axis=0)
        std_h = np.std(histories, axis=0)
        
        # Vẽ vùng mờ (độ lệch chuẩn)
        plt.fill_between(epochs, mean_h - std_h, mean_h + std_h, color=colors[idx], alpha=0.2)
        
        # Vẽ đường trung bình đậm
        plt.plot(epochs, mean_h, color=colors[idx], label=f'{emb} (Mean $\pm$ SD)', linewidth=2.5)

    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss / Accuracy', fontsize=12)
    plt.legend()
    
    output_path = save_dir / f'fig_conv_scatter_{dataset_name}.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_metrics_boxplot(df_ds: pd.DataFrame, dataset_name: str, save_dir: Path, y_min: float) -> None:
    """Vẽ 3 biểu đồ Boxplot (Accuracy, F1, AUC) cạnh nhau với Legend chung bên dưới."""
    metrics = ['Accuracy', 'F1_Score', 'AUC_Score']
    titles = ['Accuracy', 'F1 Score', 'AUC Score']
    embedding_order = ['ANGLE', 'AMPLITUDE', 'IQP']
    my_palette = sns.color_palette("Set2", n_colors=len(embedding_order))

    # Tạo subplot 1 hàng, 3 cột
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

    for i, metric in enumerate(metrics):
        if metric not in df_ds.columns:
            axes[i].set_title(f"{titles[i]} (Thiếu dữ liệu)", color='red')
            axes[i].axis('off')
            continue
            
        sns.boxplot(
            data=df_ds, x='Embedding', y=metric, hue='Embedding',
            ax=axes[i], palette=my_palette, order=embedding_order, hue_order=embedding_order,
            width=0.5, showfliers=False, boxprops={'alpha': 0.8}, legend=False
        )
        
        axes[i].set_title(titles[i], fontsize=13, fontweight='bold')
        # Tùy chỉnh giới hạn trục Y. Lưu ý: có thể tắt set_ylim nếu data của bạn vượt ngoài [0.8, 1.05]
        axes[i].set_ylim(y_min, 1.05) 
        axes[i].set_xlabel('')  
        axes[i].set_ylabel('')  
        
        # Xóa nhãn trục X để dùng Legend chung
        axes[i].set_xticklabels([])
        axes[i].tick_params(axis='x', bottom=False)

    # TẠO BẢNG KÝ HIỆU (LEGEND) CHUNG
    handles = [plt.Rectangle((0,0), 1, 1, color=my_palette[i], alpha=0.8) for i in range(len(embedding_order))]
    fig.legend(
        handles, embedding_order, title='Embedding', loc='lower center', 
        bbox_to_anchor=(0.5, 0.0), ncol=3, fontsize=12, title_fontsize=13, frameon=True
    )
    
    # Căn chỉnh để Legend không bị đè
    plt.tight_layout()
    fig.subplots_adjust(bottom=0.2) 
    
    # Lưu biểu đồ
    output_path = save_dir / f'fig_metrics_triple_box_{dataset_name}.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

def print_summary_statistics(df_ds: pd.DataFrame, dataset_name: str) -> None:
    """In bảng thống kê chi tiết Mean và Standard Deviation ra Console."""
    embedding_order = ['ANGLE', 'AMPLITUDE', 'IQP']
    
    print(f"BẢNG TỔNG HỢP KẾT QUẢ: {dataset_name.upper()}")
    print("-" * 50)
    
    for emb in embedding_order:
        df_emb = df_ds[df_ds['Embedding'] == emb]
        if df_emb.empty: 
            continue
        
        acc_m, acc_s = df_emb['Accuracy'].mean(), df_emb['Accuracy'].std()
        f1_m, f1_s = df_emb['F1_Score'].mean(), df_emb['F1_Score'].std()
        auc_m, auc_s = df_emb['AUC_Score'].mean(), df_emb['AUC_Score'].std()
        time_m, time_s = df_emb['Time_Seconds'].mean(), df_emb['Time_Seconds'].std()
        
        print(f"🔹 {emb}:")
        print(f"   - Accuracy : {acc_m:.4f} ± {acc_s:.4f}")
        print(f"   - F1 Score : {f1_m:.4f} ± {f1_s:.4f}")
        print(f"   - AUC Score: {auc_m:.4f} ± {auc_s:.4f}")
        print(f"   - Time (s) : {time_m:.2f} ± {time_s:.2f}")
    
    print("\n" + "="*80 + "\n")

csv_file = "angle_benchmark_results.csv"
save_dir = Path("images") # Thư mục lưu ảnh hiện tại
save_dir.mkdir(parents=True, exist_ok=True)

try:
    df_results = pd.read_csv(csv_file)
    
    for ds_name in df_results['Dataset'].unique():
        df_ds = df_results[df_results['Dataset'] == ds_name]
        print(f"Đang xử lý và vẽ biểu đồ cho tập dữ liệu: {ds_name}...")
        y_min = 0.0
        if ds_name == 'BANKNOTE':
            y_min = 0.8
        elif ds_name == 'BREAST_CANCER':
            y_min = 0.8
        plot_metrics_boxplot(df_ds, ds_name, save_dir, y_min)
        print_summary_statistics(df_ds, ds_name)
        plot_tradeoff_detailed(df_ds, ds_name, save_dir)
        plot_convergence_detailed(df_ds, ds_name, save_dir)
        
    print("Đã hoàn tất việc xuất biểu đồ!")
        
except FileNotFoundError:
    print(f"Không tìm thấy file '{csv_file}'. Hãy chắc chắn bạn đã tạo dữ liệu trước!")
except Exception as e:
    print(f"Có lỗi xảy ra trong quá trình xử lý: {e}")